# **Model Training — Spaceship Titanic**

**Objetivo:** Entrenar, comparar y evaluar modelos de clasificación sobre el dataset preparado en NB03.

**Cadena de análisis previa:**

| Notebook | Contribución clave |
|---|---|
| [NB01 — Exploración](01.Initial_exploration.ipynb) | 8,693 registros · 14 columnas · balanceado 50/50 · 0 duplicados |
| [NB02 — Análisis vs Target](02.Analisis_Target.ipynb) | CryoSleep (chi²=1859), Deck (chi²=392), TotalSpending_Log (r=-0.469) como features clave |
| [NB03 — Feature Engineering](03.feature_engineering.ipynb) | 8,514 filas × 35 features · escalado · guardado en `data/processed/` |

**Dataset de entrada:** `data/processed/train_features_scaled.csv`  
**Métrica principal:** Accuracy (dataset balanceado 50/50 → válida y directamente interpretable)


## **1. Librerías**


In [4]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score,
)
import joblib
import json
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


## **2. Cargar Dataset Procesado**


In [5]:
# Dataset producido por NB03: escalado, encoded, con todas las features de NB02
data_path = '../../data/processed/train_scaled.csv'

df = pd.read_csv(data_path)

target = 'Transported'
X = df.drop(columns=[target])
y = df[target]

print(f'Dataset cargado: {X.shape[0]:,} filas x {X.shape[1]} features')
print(f'Target balance: {y.mean():.1%} True | {1-y.mean():.1%} False')
print(f'\nPrimeras features: {X.columns.tolist()[:8]}')


Dataset cargado: 8,514 filas x 35 features
Target balance: 50.4% True | 49.6% False

Primeras features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNumber', 'GroupSize']


## **3. Estrategia de Validación**

- **Train/Validation split:** 80/20 con `stratify=y`
- **Cross-validation:** StratifiedKFold (5 folds) para comparación de modelos
- **Métrica principal:** Accuracy (dataset balanceado → directamente interpretable)
- **Métricas secundarias:** ROC-AUC, F1-macro


In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train : {X_train.shape[0]:,} filas')
print(f'Val   : {X_val.shape[0]:,} filas')
print(f'Train target balance: {y_train.mean():.1%}')
print(f'Val   target balance: {y_val.mean():.1%}')


Train : 6,811 filas
Val   : 1,703 filas
Train target balance: 50.4%
Val   target balance: 50.4%


## **4. Modelo Baseline**

Referencia mínima que cualquier modelo real debe superar.


In [7]:
baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train, y_train)

baseline_cv = cross_val_score(baseline, X_train, y_train, cv=cv, scoring='accuracy')
print(f'Baseline CV accuracy: {baseline_cv.mean():.4f} +/- {baseline_cv.std():.4f}')
print(f'Baseline val accuracy: {accuracy_score(y_val, baseline.predict(X_val)):.4f}')


Baseline CV accuracy: 0.5036 +/- 0.0001
Baseline val accuracy: 0.5038


## **5. Comparación de Modelos Candidatos**

Tres familias con hiperparámetros por defecto para identificar el mejor punto de partida:
- **Logistic Regression:** baseline interpretable, indica si el problema es lineal
- **Random Forest:** robusto ante no-linealidades, provee feature importance
- **Gradient Boosting:** generalmente el más preciso en tabular data


In [8]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    results[name] = {
        'cv_acc': cv_acc.mean(), 'cv_acc_std': cv_acc.std(),
        'cv_auc': cv_auc.mean(),
    }
    print(f'{name:25s}  acc={cv_acc.mean():.4f} +/-{cv_acc.std():.4f}  auc={cv_auc.mean():.4f}')


Logistic Regression        acc=0.7890 +/-0.0085  auc=0.8821
Random Forest              acc=0.8031 +/-0.0068  auc=0.8840
Gradient Boosting          acc=0.8043 +/-0.0070  auc=0.8983


In [9]:
results_df = pd.DataFrame(results).T.sort_values('cv_acc', ascending=False)

fig = px.bar(
    results_df.reset_index().rename(columns={'index': 'Modelo'}),
    x='Modelo', y='cv_acc', error_y='cv_acc_std',
    title='CV Accuracy por Modelo (5-fold)',
    labels={'cv_acc': 'CV Accuracy'},
    color='cv_acc', color_continuous_scale='Blues',
)
fig.add_hline(y=baseline_cv.mean(), line_dash='dash',
              annotation_text=f'Baseline: {baseline_cv.mean():.3f}')
fig.update_layout(height=400, showlegend=False)
fig.show()


## **6. Tuning del Mejor Modelo**

GridSearchCV sobre el modelo con mejor CV accuracy de la sección anterior.


In [10]:
best_model_name = results_df.index[0]
print(f'Modelo seleccionado para tuning: {best_model_name}')

if 'Random Forest' in best_model_name:
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
    }
    base_estimator = RandomForestClassifier(random_state=42, n_jobs=-1)
elif 'Gradient' in best_model_name:
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5],
        'learning_rate': [0.05, 0.1],
    }
    base_estimator = GradientBoostingClassifier(random_state=42)
else:
    param_grid = {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']}
    base_estimator = LogisticRegression(max_iter=1000, random_state=42)

grid_search = GridSearchCV(
    base_estimator, param_grid, cv=cv,
    scoring='accuracy', n_jobs=-1, verbose=1,
)
grid_search.fit(X_train, y_train)

print(f'\nMejores hiperparámetros: {grid_search.best_params_}')
print(f'Mejor CV accuracy     : {grid_search.best_score_:.4f}')
best_model = grid_search.best_estimator_


Modelo seleccionado para tuning: Gradient Boosting
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Mejores hiperparámetros: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Mejor CV accuracy     : 0.8121


## **7. Evaluación Final**


In [11]:
y_pred  = best_model.predict(X_val)
y_proba = best_model.predict_proba(X_val)[:, 1]

print('=' * 50)
print('EVALUACIÓN EN VALIDATION SET')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_val, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_val, y_proba):.4f}')
print()
print(classification_report(y_val, y_pred))


EVALUACIÓN EN VALIDATION SET
Accuracy : 0.8080
ROC-AUC  : 0.8974

              precision    recall  f1-score   support

       False       0.80      0.82      0.81       845
        True       0.82      0.79      0.81       858

    accuracy                           0.81      1703
   macro avg       0.81      0.81      0.81      1703
weighted avg       0.81      0.81      0.81      1703



In [12]:
cm = confusion_matrix(y_val, y_pred)
fig = px.imshow(
    cm, text_auto=True, color_continuous_scale='Blues',
    labels={'x': 'Prediccion', 'y': 'Real'},
    x=['False', 'True'], y=['False', 'True'],
    title='Confusion Matrix — Validation Set',
)
fig.update_layout(height=400, width=500)
fig.show()


### **7.1 Feature Importance**

> Verificamos que las features de mayor poder discriminativo en NB02
> (CryoSleep, Deck, TotalSpending_Log) aparezcan entre las más importantes del modelo.


In [13]:
if hasattr(best_model, 'feature_importances_'):
    fi = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model.feature_importances_,
    }).sort_values('importance', ascending=False).head(20)
elif hasattr(best_model, 'coef_'):
    fi = pd.DataFrame({
        'feature': X.columns,
        'importance': abs(best_model.coef_[0]),
    }).sort_values('importance', ascending=False).head(20)

fig = px.bar(
    fi, x='importance', y='feature', orientation='h',
    title='Top 20 Features por Importancia',
    labels={'importance': 'Importancia', 'feature': 'Feature'},
    color='importance', color_continuous_scale='Viridis',
)
fig.update_layout(height=600, showlegend=False)
fig.show()


## **8. Guardar Modelo**


In [14]:
model_dir = '../../models'
os.makedirs(model_dir, exist_ok=True)

model_path = f'{model_dir}/best_model.pkl'
joblib.dump(best_model, model_path)

metadata = {
    'model': best_model_name,
    'params': grid_search.best_params_,
    'cv_accuracy': round(grid_search.best_score_, 4),
    'val_accuracy': round(accuracy_score(y_val, y_pred), 4),
    'val_roc_auc': round(roc_auc_score(y_val, y_proba), 4),
    'n_features': int(X.shape[1]),
    'n_train_samples': int(X_train.shape[0]),
}
with open(f'{model_dir}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Modelo guardado: {model_path}')
print(f'Metadatos: {metadata}')


Modelo guardado: ../../models/best_model.pkl
Metadatos: {'model': 'Gradient Boosting', 'params': {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}, 'cv_accuracy': 0.8121, 'val_accuracy': 0.808, 'val_roc_auc': 0.8974, 'n_features': 35, 'n_train_samples': 6811}


## **9. Resumen**


In [15]:
print('=' * 60)
print('RESUMEN DE ENTRENAMIENTO')
print('=' * 60)
print(f'  Modelo            : {best_model_name}')
print(f'  Hiperparámetros   : {grid_search.best_params_}')
print(f'  CV Accuracy 5fold : {grid_search.best_score_:.4f}')
print(f'  Val Accuracy      : {accuracy_score(y_val, y_pred):.4f}')
print(f'  Val ROC-AUC       : {roc_auc_score(y_val, y_proba):.4f}')
print(f'  Baseline (dummy)  : {baseline_cv.mean():.4f}')
print(f'  Ganancia vs base  : +{accuracy_score(y_val, y_pred) - baseline_cv.mean():.4f}')
print('=' * 60)
print()
print('Modelo listo → ver src/models/ para integración en pipeline')


RESUMEN DE ENTRENAMIENTO
  Modelo            : Gradient Boosting
  Hiperparámetros   : {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
  CV Accuracy 5fold : 0.8121
  Val Accuracy      : 0.8080
  Val ROC-AUC       : 0.8974
  Baseline (dummy)  : 0.5036
  Ganancia vs base  : +0.3044

Modelo listo → ver src/models/ para integración en pipeline
